# Notebook 05: Dashboard JSON Export Pipeline

**Project:** Shopee App Review Intelligence Dashboard  
**Input:** `data/processed/shopee_reviews_final.csv` (85.499 baris)  
**Output Location:** `outputs/json/` & `dashboard/public/data/`

---
## Tujuan
Menghasilkan 6 file JSON terstruktur yang siap dikonsumsi secara independen oleh frontend React Dashboard (Fase 6):
1. `overview.json` — Ringkasan KPI global, rating distribution, & proporsi sentimen.
2. `sentiment_trend.json` — Trend bulanan volume sentimen & rata-rata rating.
3. `version_quality.json` — Skor kualitas & % review negatif per versi aplikasi (`reviewCreatedVersion`).
4. `topics.json` — Struct 8 topik keluhan negatif beserta kata kunci & contoh review.
5. `top_reviews.json` — Top 10 review ber-`thumbsUpCount` tertinggi per sentimen.
6. `wordcloud_data.json` — Top 50 frekuensi kata terbanyak per sentimen.


In [1]:
import os, json, shutil
from collections import Counter
import pandas as pd
import numpy as np

# Create directories
os.makedirs("../outputs/json", exist_ok=True)
os.makedirs("../dashboard/public/data", exist_ok=True)

# Load Final Dataset
df_path = os.path.join("..", "data", "processed", "shopee_reviews_final.csv")
df = pd.read_csv(df_path)
df['clean_content'] = df['clean_content'].fillna('').astype(str)
df['content']       = df['content'].fillna('').astype(str)
df['userName']      = df['userName'].fillna('Pengguna Shopee').astype(str)

print(f"Loaded final dataset: {len(df):,} rows")


Loaded final dataset: 85,499 rows


In [2]:
# 1. Export overview.json
total_reviews = len(df)
avg_rating = round(float(df['score'].mean()), 2)
sentiment_counts = df['predicted_sentiment'].value_counts().to_dict()
sentiment_pcts = {k: round(v / total_reviews * 100, 2) for k, v in sentiment_counts.items()}
rating_counts = {str(k): int(v) for k, v in df['score'].value_counts().sort_index().to_dict().items()}

overview_data = {
    "total_reviews": total_reviews,
    "avg_rating": avg_rating,
    "sentiment_counts": {
        "positif": int(sentiment_counts.get("positif", 0)),
        "negatif": int(sentiment_counts.get("negatif", 0)),
        "netral": int(sentiment_counts.get("netral", 0))
    },
    "sentiment_percentages": {
        "positif": sentiment_pcts.get("positif", 0.0),
        "negatif": sentiment_pcts.get("negatif", 0.0),
        "netral": sentiment_pcts.get("netral", 0.0)
    },
    "rating_counts": rating_counts
}

with open("../outputs/json/overview.json", "w", encoding="utf-8") as f:
    json.dump(overview_data, f, indent=2, ensure_ascii=False)

print("overview.json exported:")
print(json.dumps(overview_data, indent=2))


overview.json exported:
{
  "total_reviews": 85499,
  "avg_rating": 3.09,
  "sentiment_counts": {
    "positif": 36796,
    "negatif": 38506,
    "netral": 10197
  },
  "sentiment_percentages": {
    "positif": 43.04,
    "negatif": 45.04,
    "netral": 11.93
  },
  "rating_counts": {
    "1": 28309,
    "2": 8528,
    "3": 8540,
    "4": 7579,
    "5": 32543
  }
}


In [3]:
# 2. Export sentiment_trend.json
df['at_dt'] = pd.to_datetime(df['at'], errors='coerce')
df['month'] = df['at_dt'].dt.strftime('%Y-%m')

df_trend = df[df['month'].notna()].copy()
grouped_trend = df_trend.groupby(['month', 'predicted_sentiment']).size().unstack(fill_value=0)
avg_ratings_monthly = df_trend.groupby('month')['score'].mean().round(2)

trend_list = []
for month in sorted(grouped_trend.index):
    row = grouped_trend.loc[month]
    pos = int(row.get('positif', 0)); neu = int(row.get('netral', 0)); neg = int(row.get('negatif', 0))
    tot = pos + neu + neg
    trend_list.append({
        "month": month,
        "total": tot,
        "positif": pos, "netral": neu, "negatif": neg,
        "avg_rating": float(avg_ratings_monthly.get(month, 0.0)),
        "negative_rate": round(neg / tot * 100, 2) if tot > 0 else 0.0
    })

with open("../outputs/json/sentiment_trend.json", "w", encoding="utf-8") as f:
    json.dump(trend_list, f, indent=2, ensure_ascii=False)

print(f"sentiment_trend.json exported ({len(trend_list)} months)")
print(f"Sample month: {trend_list[-1]}")


sentiment_trend.json exported (76 months)
Sample month: {'month': '2024-12', 'total': 5142, 'positif': 3262, 'netral': 312, 'negatif': 1568, 'avg_rating': 3.73, 'negative_rate': 30.49}


In [4]:
# 3. Export version_quality.json
df_ver = df[df['reviewCreatedVersion'].notna() & (df['reviewCreatedVersion'].astype(str).str.strip() != '')].copy()
df_ver['version'] = df_ver['reviewCreatedVersion'].astype(str)

ver_counts = df_ver['version'].value_counts()
valid_versions = ver_counts[ver_counts >= 30].index

ver_list = []
for ver in valid_versions:
    sub = df_ver[df_ver['version'] == ver]
    tot = len(sub)
    avg_r = round(float(sub['score'].mean()), 2)
    s_counts = sub['predicted_sentiment'].value_counts()
    neg_c = int(s_counts.get('negatif', 0)); pos_c = int(s_counts.get('positif', 0))
    ver_list.append({
        "version": ver,
        "review_count": tot,
        "avg_rating": avg_r,
        "negative_count": neg_c,
        "negative_pct": round(neg_c / tot * 100, 2),
        "positive_pct": round(pos_c / tot * 100, 2)
    })

ver_list.sort(key=lambda x: x['review_count'], reverse=True)

with open("../outputs/json/version_quality.json", "w", encoding="utf-8") as f:
    json.dump(ver_list, f, indent=2, ensure_ascii=False)

print(f"version_quality.json exported ({len(ver_list)} app versions)")


version_quality.json exported (259 app versions)


In [5]:
# 4. Export top_reviews.json
top_reviews_data = {}
for sent in ['positif', 'netral', 'negatif']:
    sub_s = df[df['predicted_sentiment'] == sent].sort_values(by=['thumbsUpCount', 'score'], ascending=[False, sent == 'positif'])
    top10 = sub_s.head(10)
    items = []
    for _, r in top10.iterrows():
        items.append({
            "reviewId": str(r.get('reviewId', '')),
            "userName": str(r.get('userName', 'Pengguna Shopee')),
            "content": str(r.get('content', '')),
            "score": int(r.get('score', 0)),
            "thumbsUpCount": int(r.get('thumbsUpCount', 0)),
            "at": str(r.get('at', '')),
            "predicted_sentiment": sent,
            "topic_label": str(r.get('topic_label', '')) if pd.notna(r.get('topic_label')) else None
        })
    top_reviews_data[sent] = items

with open("../outputs/json/top_reviews.json", "w", encoding="utf-8") as f:
    json.dump(top_reviews_data, f, indent=2, ensure_ascii=False)

print("top_reviews.json exported (top 10 per sentiment)")


top_reviews.json exported (top 10 per sentiment)


In [6]:
# 5. Export wordcloud_data.json
wordcloud_data = {}
for sent in ['positif', 'netral', 'negatif']:
    sub_s = df[df['predicted_sentiment'] == sent]
    all_words = []
    for text in sub_s['clean_content']:
        words = [w for w in text.split() if len(w) > 2]
        all_words.extend(words)
    counter = Counter(all_words)
    top50 = [{"text": word, "value": count} for word, count in counter.most_common(50)]
    wordcloud_data[sent] = top50

with open("../outputs/json/wordcloud_data.json", "w", encoding="utf-8") as f:
    json.dump(wordcloud_data, f, indent=2, ensure_ascii=False)

print("wordcloud_data.json exported (top 50 words per sentiment)")


wordcloud_data.json exported (top 50 words per sentiment)


In [7]:
# Validation & Copy to Dashboard Public Data
json_files = ['overview.json', 'sentiment_trend.json', 'version_quality.json', 'topics.json', 'top_reviews.json', 'wordcloud_data.json']

print("Validating & copying JSON files to dashboard/public/data/...")
for jf in json_files:
    src = os.path.join("..", "outputs", "json", jf)
    dst = os.path.join("..", "dashboard", "public", "data", jf)
    with open(src, "r", encoding="utf-8") as f:
        data = json.load(f)
    shutil.copy2(src, dst)
    size_kb = round(os.path.getsize(src) / 1024, 2)
    print(f"  ✓ {jf:<22} ({size_kb:>6.1f} KB) -> {dst}")

print("\nALL DASHBOARD JSON EXPORTS COMPLETED!")


Validating & copying JSON files to dashboard/public/data/...
  ✓ overview.json          (   0.3 KB) -> ..\dashboard\public\data\overview.json
  ✓ sentiment_trend.json   (  12.6 KB) -> ..\dashboard\public\data\sentiment_trend.json
  ✓ version_quality.json   (  43.1 KB) -> ..\dashboard\public\data\version_quality.json
  ✓ topics.json            (   3.6 KB) -> ..\dashboard\public\data\topics.json
  ✓ top_reviews.json       (  19.6 KB) -> ..\dashboard\public\data\top_reviews.json
  ✓ wordcloud_data.json    (   8.9 KB) -> ..\dashboard\public\data\wordcloud_data.json

ALL DASHBOARD JSON EXPORTS COMPLETED!
